# Modélisation Deep Learning - Amazon Sentiment

Ce notebook implémente un modèle de classification de sentiment utilisant TensorFlow/Keras.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, GlobalMaxPooling1D
import os

## Chargement et Préparation des données

In [ ]:
df = pd.read_csv(os.path.join('..', 'data', 'amazon_reviews.csv'))

# Encodage des labels
le = LabelEncoder()
df['sentiment_encoded'] = le.fit_transform(df['sentiment'])
num_classes = len(le.classes_)

X = df['review_body'].values
y = df['sentiment_encoded'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Tokenisation et Padding

In [ ]:
max_words = 5000
max_len = 50

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len)

## Construction du Modèle

In [ ]:
model = Sequential([
    Embedding(max_words, 64, input_length=max_len),
    LSTM(64, return_sequences=True),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## Entraînement

In [ ]:
history = model.fit(X_train_seq, y_train, epochs=5, batch_size=32, 
                    validation_data=(X_test_seq, y_test), verbose=1)

## Visualisation des Performances avec Plotly

In [ ]:
history_df = pd.DataFrame(history.history)

fig_acc = go.Figure()
fig_acc.add_trace(go.Scatter(y=history_df['accuracy'], name='Train Accuracy'))
fig_acc.add_trace(go.Scatter(y=history_df['val_accuracy'], name='Val Accuracy'))
fig_acc.update_layout(title='Précision du Modèle', xaxis_title='Epoque', yaxis_title='Précision')
fig_acc.show()

fig_loss = go.Figure()
fig_loss.add_trace(go.Scatter(y=history_df['loss'], name='Train Loss'))
fig_loss.add_trace(go.Scatter(y=history_df['val_loss'], name='Val Loss'))
fig_loss.update_layout(title='Perte du Modèle', xaxis_title='Epoque', yaxis_title='Perte')
fig_loss.show()